# Exercise 02 — JSON Schema
### 🔍 Validating sensor data before it enters the system

In a real IoT network, sensors are unreliable. They can send:
- Missing fields
- Wrong types (a string instead of a number)
- Out-of-range values

**JSON Schema** is the answer — it is a vocabulary for describing the structure
a JSON document must have. Think of it as the IoT equivalent of XML's DTD,
but for JSON (as your lecture notes warn: *JSON has no built-in validation!*).

In this exercise you will:
- Write a JSON Schema for a sensor reading
- Validate good and bad messages
- Understand common validation errors

---

In [ ]:
# Install if needed:  pip install jsonschema
import json
from jsonschema import validate, ValidationError

## 1️⃣  Define the schema

A schema is itself a JSON object that describes what a valid message looks like.
Here we define exactly what a sensor reading from SmartCity Turin must contain.

In [ ]:
sensor_schema = {
    "type": "object",
    "required": ["sensor_id", "location", "active", "readings"],
    "properties": {
        "sensor_id": {
            "type": "string",
            "pattern": "^TRN-[0-9]{3}$",          # must match TRN-001, TRN-042, etc.
            "description": "Unique sensor identifier"
        },
        "location": {
            "type": "string",
            "minLength": 3
        },
        "active": {
            "type": "boolean"
        },
        "readings": {
            "type": "object",
            "required": ["temperature_c", "humidity_pct", "pm25_ugm3"],
            "properties": {
                "temperature_c": {"type": "number", "minimum": -40, "maximum": 85},
                "humidity_pct":  {"type": "number", "minimum": 0,   "maximum": 100},
                "pm25_ugm3":     {"type": "number", "minimum": 0}
            },
            "additionalProperties": False   # no unexpected extra fields
        }
    },
    "additionalProperties": False
}

print("Schema defined ✓")

## 2️⃣  Validate a correct message

`validate()` raises an exception if invalid. If nothing is raised — it's valid!

In [ ]:
good_message = {
    "sensor_id": "TRN-001",
    "location": "Piazza Castello",
    "active": True,
    "readings": {
        "temperature_c": 22.4,
        "humidity_pct": 58,
        "pm25_ugm3": 12.1
    }
}

try:
    validate(instance=good_message, schema=sensor_schema)
    print("✅ Message is valid!")
except ValidationError as e:
    print(f"❌ Invalid: {e.message}")

## 3️⃣  Break it on purpose — spot the errors

Each cell below has **one bug**. Run it, read the error message, then fix it.

In [ ]:
# BUG A: temperature sent as a string instead of a number
bad_a = {
    "sensor_id": "TRN-002",
    "location": "Lingotto",
    "active": True,
    "readings": {
        "temperature_c": "hot",   # <-- wrong type!
        "humidity_pct": 60,
        "pm25_ugm3": 15
    }
}
try:
    validate(instance=bad_a, schema=sensor_schema)
    print("✅ Valid")
except ValidationError as e:
    print(f"❌ Bug A caught: {e.message}")

In [ ]:
# BUG B: missing required field
bad_b = {
    "sensor_id": "TRN-003",
    "active": True,
    # location is missing!
    "readings": {
        "temperature_c": 21.0,
        "humidity_pct": 70,
        "pm25_ugm3": 35
    }
}
try:
    validate(instance=bad_b, schema=sensor_schema)
    print("✅ Valid")
except ValidationError as e:
    print(f"❌ Bug B caught: {e.message}")

In [ ]:
# BUG C: sensor_id doesn't match the pattern TRN-XXX
bad_c = {
    "sensor_id": "SENSOR_42",   # <-- wrong format!
    "location": "Mirafiori",
    "active": False,
    "readings": {
        "temperature_c": 19.0,
        "humidity_pct": 80,
        "pm25_ugm3": 10
    }
}
try:
    validate(instance=bad_c, schema=sensor_schema)
    print("✅ Valid")
except ValidationError as e:
    print(f"❌ Bug C caught: {e.message}")

In [ ]:
# BUG D: humidity out of physical range
bad_d = {
    "sensor_id": "TRN-004",
    "location": "Parco del Po",
    "active": True,
    "readings": {
        "temperature_c": 20.5,
        "humidity_pct": 150,   # <-- impossible value!
        "pm25_ugm3": 9
    }
}
try:
    validate(instance=bad_d, schema=sensor_schema)
    print("✅ Valid")
except ValidationError as e:
    print(f"❌ Bug D caught: {e.message}")

## 4️⃣  Build a validation gateway

In a real IoT pipeline, a **gateway** validates every incoming message.
Valid ones go to storage; invalid ones go to an error queue.

In [ ]:
incoming_messages = [
    {"sensor_id": "TRN-001", "location": "Piazza Castello", "active": True,  "readings": {"temperature_c": 22.4, "humidity_pct": 58,  "pm25_ugm3": 12.1}},
    {"sensor_id": "TRN-002", "location": "Lingotto",        "active": True,  "readings": {"temperature_c": "hot","humidity_pct": 60,  "pm25_ugm3": 15}},
    {"sensor_id": "TRN-003", "location": "Mirafiori",       "active": False, "readings": {"temperature_c": 21.0, "humidity_pct": 70,  "pm25_ugm3": 35}},
    {"sensor_id": "TRN-004", "location": "Parco del Po",    "active": True,  "readings": {"temperature_c": 20.5, "humidity_pct": 150, "pm25_ugm3": 9}},
    {"sensor_id": "TRN-005", "location": "Vittorio",        "active": True,  "readings": {"temperature_c": 23.1, "humidity_pct": 61,  "pm25_ugm3": 18}},
]

accepted = []
rejected = []

for msg in incoming_messages:
    try:
        validate(instance=msg, schema=sensor_schema)
        accepted.append(msg)
    except ValidationError as e:
        rejected.append({"message": msg, "reason": e.message})

print(f"✅ Accepted: {len(accepted)}")
print(f"❌ Rejected: {len(rejected)}")
print()
for r in rejected:
    print(f"  Sensor {r['message'].get('sensor_id','?')} → {r['reason']}")

---
## 🏆 Challenges

1. Add a new optional field `"battery_pct"` to the schema (number, 0–100). Make it optional but validated when present.
2. Add an `"enum"` constraint on a new `"unit"` field inside readings that only accepts `"metric"` or `"imperial"`.
3. Write a function `is_valid(msg)` that returns `True/False` instead of raising an exception.
4. Extend the gateway to save accepted messages to `accepted.json` and rejected ones to `errors.json`.

In [ ]:
# Your code here
